# Issue #46: Feature experiments and error analysis

This notebook reproduces the selected full-feature HistGradientBoosting candidate and tests optional experiment O1 under the frozen October evaluation design. Predictors remain anonymous: coefficients or importance values do not establish business meaning or causal effects. The protected test partition is excluded before any feature analysis, model fitting, prediction, or scoring.

## 1. Environment, paths, and immutable inputs

The run uses modeling data `OCT-MD-001`, split `OCT-SPLIT-001`, seed 42, and the repository-pinned scikit-learn 1.6.1 environment.

In [1]:
from pathlib import Path
import hashlib
import json
import platform
import subprocess
import time

import nbformat
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

ROOT_OVERRIDE = None  # Colab users may set this to the repository root.
if ROOT_OVERRIDE is not None:
    ROOT = Path(ROOT_OVERRIDE).expanduser().resolve()
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    ROOT = next(path for path in candidates if (path / 'data/allstate_claims_data.csv').is_file())
BUNDLE = ROOT / 'notebooks/ragib-nehal/oct-notebooks/feature-experiments-review'
DATA = ROOT / 'data/allstate_claims_data.csv'
MANIFEST = ROOT / 'notebooks/final-deliverables/October/Gate 1/partition_manifest.csv'
ISSUE45 = ROOT / 'notebooks/final-deliverables/October/Gste 3/controlled_model_comparison/model_comparison_results'
NOTEBOOK_PATH = BUNDLE / 'error_analysis.ipynb'
SOURCE_SHA256 = '74037cb248a1064e4d578692a4f4e5d8492ed1b2033daf643496e1b68b14ae03'
MANIFEST_SHA256 = 'de10b0e17ad56174a7b126352a0a41b3b3d477bbddd65f37b87a6235f582a64b'
WORKFLOW_RUN = 'OCT-RN-FEATURE-REVIEW-001'
SEED = 42
RARE_MIN_FREQUENCY = 132

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

assert sha256(DATA) == SOURCE_SHA256
assert sha256(MANIFEST) == MANIFEST_SHA256
git_commit = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=ROOT, capture_output=True, text=True, check=True).stdout.strip()
git_dirty_at_start = bool(subprocess.run(['git', 'status', '--porcelain'], cwd=ROOT, capture_output=True, text=True, check=True).stdout.strip())
print({'repository_root': '.', 'workflow_run': WORKFLOW_RUN, 'git_commit': git_commit, 'git_dirty_at_start': git_dirty_at_start})

{'root': '/Users/ragibn/Library/CloudStorage/OneDrive-Personal/Documents/Programs/BTT - AI Studio Fall/Allstate-1A-predicting-auto-claims-severity', 'workflow_run': 'OCT-RN-FEATURE-REVIEW-001', 'git_commit': '0693f5d5a75569778590fdf081e0b1695e2b38b2', 'git_dirty_at_start': False}


## 2. Frozen train/validation data and training-only feature rule

Partition membership is filtered through the manifest before source rows are retained. Dominant-level shares and the O1 exclusion set are calculated from training rows only.

In [2]:
manifest = pd.read_csv(MANIFEST)
assert manifest['partition'].value_counts().to_dict() == {'train': 131_822, 'validation': 28_248, 'test': 28_248}
model_manifest = manifest.loc[manifest['partition'].isin(['train', 'validation'])].copy()
model_ids = set(model_manifest['id'])
source = pd.read_csv(DATA)
model_source = source.loc[source['id'].isin(model_ids)].copy()
del source
modeling = model_source.merge(model_manifest, on='id', validate='one_to_one')
assert set(modeling['partition']) == {'train', 'validation'}
train = modeling.loc[modeling['partition'] == 'train'].reset_index(drop=True)
validation = modeling.loc[modeling['partition'] == 'validation'].reset_index(drop=True)
assert (len(train), len(validation)) == (131_822, 28_248)
assert set(train['cv_fold'].astype(int)) == set(range(5))

CAT = [f'cat{i}' for i in range(1, 117)]
CONT = [f'cont{i}' for i in range(1, 15)]
FULL_FEATURES = CAT + CONT
dominant_share = train[CAT].apply(lambda column: column.value_counts(dropna=False, normalize=True).iloc[0])
excluded_cat = dominant_share.loc[dominant_share > 0.99].index.tolist()
EXPECTED_EXCLUDED = ['cat15', 'cat17', 'cat18', 'cat19', 'cat20', 'cat21', 'cat22', 'cat32', 'cat33', 'cat34', 'cat35', 'cat42', 'cat46', 'cat47', 'cat48', 'cat51', 'cat55', 'cat56', 'cat58', 'cat59', 'cat60', 'cat61', 'cat62', 'cat63', 'cat64', 'cat67', 'cat68', 'cat69', 'cat70', 'cat77', 'cat78']
assert excluded_cat == EXPECTED_EXCLUDED
REDUCED_CAT = [feature for feature in CAT if feature not in excluded_cat]
REDUCED_FEATURES = REDUCED_CAT + CONT
assert (len(FULL_FEATURES), len(REDUCED_FEATURES)) == (130, 99)

train_p90 = float(train['loss'].quantile(0.90))
train_p99 = float(train['loss'].quantile(0.99))
issue45_predictions = pd.read_csv(ISSUE45 / 'validation_predictions.csv').set_index('id')
validation_indexed = validation.set_index('id')
assert set(issue45_predictions.index) == set(validation_indexed.index)
issue45_predictions = issue45_predictions.loc[validation_indexed.index]
assert np.allclose(issue45_predictions['loss'], validation_indexed['loss'])
print({'excluded_fields': len(excluded_cat), 'full_features': len(FULL_FEATURES), 'reduced_features': len(REDUCED_FEATURES), 'train_p90': train_p90, 'train_p99': train_p99})

{'excluded_fields': 31, 'full_features': 130, 'reduced_features': 99, 'train_p90': 6401.738, 'train_p99': 14024.143299999967}


## 3. Fold-safe E02 reproduction and O1 comparison

Each fold constructs a fresh ordinal encoder and HGB model. The O1 validation score is consulted once after its training-fold result is available. The full E02 validation score is a reproduction of an already selected configuration.

In [3]:
MODEL_SETTINGS = {
    'loss': 'absolute_error',
    'learning_rate': 0.1,
    'max_leaf_nodes': 63,
    'min_samples_leaf': 50,
    'max_iter': 200,
    'early_stopping': True,
    'random_state': SEED,
}

def make_hgb(categorical_features):
    features = categorical_features + CONT
    preprocessor = ColumnTransformer([
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=np.nan, min_frequency=RARE_MIN_FREQUENCY), categorical_features),
        ('cont', 'passthrough', CONT),
    ])
    categorical_mask = np.array([True] * len(categorical_features) + [False] * len(CONT))
    model = HistGradientBoostingRegressor(categorical_features=categorical_mask, **MODEL_SETTINGS)
    return features, Pipeline([('prep', preprocessor), ('model', model)])

def cross_validate_hgb(categorical_features):
    features, _ = make_hgb(categorical_features)
    fold_maes = []
    started = time.perf_counter()
    for fold in range(5):
        fit_rows = train.loc[train['cv_fold'] != fold]
        hold_rows = train.loc[train['cv_fold'] == fold]
        _, pipeline = make_hgb(categorical_features)
        pipeline.fit(fit_rows[features], fit_rows['loss'])
        prediction = pipeline.predict(hold_rows[features])
        assert np.isfinite(prediction).all()
        fold_maes.append(float(mean_absolute_error(hold_rows['loss'], prediction)))
    return fold_maes, time.perf_counter() - started

full_fold_maes, full_cv_seconds = cross_validate_hgb(CAT)
o1_fold_maes, o1_cv_seconds = cross_validate_hgb(REDUCED_CAT)

def fit_validation(categorical_features):
    features, pipeline = make_hgb(categorical_features)
    started = time.perf_counter()
    pipeline.fit(train[features], train['loss'])
    prediction = pipeline.predict(validation[features])
    elapsed = time.perf_counter() - started
    assert np.isfinite(prediction).all()
    return prediction, elapsed

pred_e02_full, full_fit_seconds = fit_validation(CAT)
pred_o1_reduced, o1_fit_seconds = fit_validation(REDUCED_CAT)
print({'full_cv_maes': full_fold_maes, 'o1_cv_maes': o1_fold_maes})

{'full_cv_maes': [1177.6925687563246, 1168.48538226761, 1167.557816484496, 1179.2865044616622, 1164.5157530149545], 'o1_cv_maes': [1178.1728307115718, 1171.0599331462204, 1167.6138674860067, 1182.0221815831846, 1164.0301046294792]}


## 4. Decision, error diagnostics, and evidence exports

O1 is selected only when both CV and validation MAE improve by at least 1%. Severity results remain diagnostics rather than additional selection metrics.

In [4]:
full_cv_mean = float(np.mean(full_fold_maes))
o1_cv_mean = float(np.mean(o1_fold_maes))
full_validation_mae = float(mean_absolute_error(validation['loss'], pred_e02_full))
o1_validation_mae = float(mean_absolute_error(validation['loss'], pred_o1_reduced))
cv_gain_pct = 100 * (full_cv_mean - o1_cv_mean) / full_cv_mean
validation_gain_pct = 100 * (full_validation_mae - o1_validation_mae) / full_validation_mae
selected_run = 'O1' if cv_gain_pct >= 1.0 and validation_gain_pct >= 1.0 else 'E02-full'
feature_decision = 'exclude_near_constant_fields' if selected_run == 'O1' else 'retain_all_130_predictors'

comparison = pd.DataFrame([
    {'run_id': 'E02-full', 'role': 'selected-model reproduction', 'feature_count': 130, 'excluded_feature_count': 0, 'cv_folds': 5, 'cv_mae_mean': full_cv_mean, 'cv_mae_sd': float(np.std(full_fold_maes)), 'validation_mae': full_validation_mae, 'validation_claims': len(validation), 'cv_seconds': full_cv_seconds, 'final_fit_seconds': full_fit_seconds, 'cv_gain_vs_full_pct': 0.0, 'validation_gain_vs_full_pct': 0.0, 'selected': selected_run == 'E02-full'},
    {'run_id': 'O1', 'role': 'optional near-constant exclusion', 'feature_count': 99, 'excluded_feature_count': 31, 'cv_folds': 5, 'cv_mae_mean': o1_cv_mean, 'cv_mae_sd': float(np.std(o1_fold_maes)), 'validation_mae': o1_validation_mae, 'validation_claims': len(validation), 'cv_seconds': o1_cv_seconds, 'final_fit_seconds': o1_fit_seconds, 'cv_gain_vs_full_pct': cv_gain_pct, 'validation_gain_vs_full_pct': validation_gain_pct, 'selected': selected_run == 'O1'},
])
comparison.to_csv(BUNDLE / 'feature_experiment_comparison.csv', index=False)

decisions = pd.DataFrame([
    {'experiment_id': 'O1', 'status': 'completed', 'hypothesis': 'Removing training-defined near-constant categorical fields may preserve accuracy with fewer inputs.', 'decision': feature_decision, 'reason': f'Predeclared rule requires at least 1% improvement in both CV and validation MAE; observed gains were {cv_gain_pct:.4f}% and {validation_gain_pct:.4f}%.', 'excluded_features': '|'.join(excluded_cat), 'next_gate': 'independent review'},
    {'experiment_id': 'O2', 'status': 'deferred', 'hypothesis': 'Reducing correlated continuous pairs may change MAE little.', 'decision': 'defer', 'reason': 'Correlation alone does not justify field deletion, and regularization/tree handling already addresses redundancy.', 'excluded_features': '', 'next_gate': 'November handoff'},
    {'experiment_id': 'O3', 'status': 'deferred', 'hypothesis': 'Fold-safe data-driven selection may retain accuracy with fewer fields.', 'decision': 'defer', 'reason': 'Avoid another feature-selection surface after limited validation use.', 'excluded_features': '', 'next_gate': 'November handoff'},
    {'experiment_id': 'O4', 'status': 'deferred', 'hypothesis': 'A log1p target may reduce linear-model MAE after back-transformation.', 'decision': 'defer', 'reason': 'This changes the target transformation rather than feature inclusion and is not required for Gate 3.', 'excluded_features': '', 'next_gate': 'November handoff'},
])
decisions.to_csv(BUNDLE / 'feature_decisions.csv', index=False)

predictions = pd.DataFrame({
    'id': validation['id'].to_numpy(),
    'loss': validation['loss'].to_numpy(),
    'pred_training_mean': issue45_predictions['pred_training_mean'].to_numpy(),
    'pred_training_median': issue45_predictions['pred_training_median'].to_numpy(),
    'pred_ridge': issue45_predictions['pred_ridge'].to_numpy(),
    'pred_e02_full': pred_e02_full,
    'pred_o1_reduced': pred_o1_reduced,
})
predictions.to_csv(BUNDLE / 'validation_predictions.csv', index=False)

segments = [
    ('loss < train p90', predictions['loss'] < train_p90),
    ('train p90 <= loss < p99', (predictions['loss'] >= train_p90) & (predictions['loss'] < train_p99)),
    ('loss >= train p99', predictions['loss'] >= train_p99),
    ('all validation claims', pd.Series(True, index=predictions.index)),
]
model_columns = {
    'training_mean': 'pred_training_mean',
    'training_median': 'pred_training_median',
    'ridge': 'pred_ridge',
    'e02_full': 'pred_e02_full',
    'o1_reduced': 'pred_o1_reduced',
}
diagnostic_rows = []
for model, prediction_column in model_columns.items():
    for segment, mask in segments:
        actual = predictions.loc[mask, 'loss'].to_numpy()
        predicted = predictions.loc[mask, prediction_column].to_numpy()
        signed_error = predicted - actual
        absolute_error = np.abs(signed_error)
        diagnostic_rows.append({
            'model': model, 'partition': 'validation', 'segment': segment, 'support_claims': len(actual),
            'mae': float(np.mean(absolute_error)), 'mean_signed_error': float(np.mean(signed_error)),
            'median_absolute_error': float(np.median(absolute_error)),
            'p90_absolute_error': float(np.quantile(absolute_error, 0.90)),
            'p99_absolute_error': float(np.quantile(absolute_error, 0.99)),
            'underprediction_rate': float(np.mean(predicted < actual)),
            'negative_predictions': int((predicted < 0).sum()),
            'non_finite_predictions': int((~np.isfinite(predicted)).sum()),
        })
diagnostics = pd.DataFrame(diagnostic_rows)
diagnostics.to_csv(BUNDLE / 'error_diagnostics.csv', index=False)

selected_column = 'pred_o1_reduced' if selected_run == 'O1' else 'pred_e02_full'
largest = predictions[['id', 'loss', selected_column]].copy()
largest = largest.rename(columns={selected_column: 'prediction'})
largest['signed_error'] = largest['prediction'] - largest['loss']
largest['abs_error'] = largest['signed_error'].abs()
largest['selected_run'] = selected_run
largest.nlargest(25, 'abs_error').to_csv(BUNDLE / 'largest_errors.csv', index=False)
print(comparison.to_string(index=False))
print({'selected_run': selected_run, 'feature_decision': feature_decision})

  run_id                             role  feature_count  excluded_feature_count  cv_folds  cv_mae_mean  cv_mae_sd  validation_mae  validation_claims  cv_seconds  final_fit_seconds  cv_gain_vs_full_pct  validation_gain_vs_full_pct  selected
E02-full      selected-model reproduction            130                       0         5  1171.507605   5.871724     1151.244420              28248   38.239468           9.471361             0.000000                     0.000000      True
      O1 optional near-constant exclusion             99                      31         5  1172.579784   6.640909     1152.503806              28248   29.656658           7.131752            -0.091521                    -0.109393     False
{'selected_run': 'E02-full', 'feature_decision': 'retain_all_130_predictors'}


## 5. Run receipt

The receipt binds the evidence to immutable inputs, notebook code, settings, environment versions, and generated output hashes. Independent review and Gate 3 approval remain separate pending actions.

In [5]:
notebook = nbformat.read(NOTEBOOK_PATH, as_version=4)
notebook_code = '\n\n'.join(cell.source for cell in notebook.cells if cell.cell_type == 'code')
evidence_names = ['feature_experiment_comparison.csv', 'feature_decisions.csv', 'validation_predictions.csv', 'error_diagnostics.csv', 'largest_errors.csv']
receipt = {
    'workflow_run': WORKFLOW_RUN,
    'modeling_data_version': 'OCT-MD-001',
    'split_version': 'OCT-SPLIT-001',
    'seed': SEED,
    'source_sha256': SOURCE_SHA256,
    'manifest_sha256': MANIFEST_SHA256,
    'notebook_code_sha256': hashlib.sha256(notebook_code.encode('utf-8')).hexdigest(),
    'code_version_commit': git_commit,
    'git_dirty_at_start': git_dirty_at_start,
    'partition_counts_used': {'train': len(train), 'validation': len(validation), 'test': 'excluded'},
    'cv_folds': 5,
    'model_settings': {**MODEL_SETTINGS, 'rare_min_frequency': RARE_MIN_FREQUENCY},
    'feature_counts': {'E02-full': 130, 'O1': 99},
    'excluded_features': excluded_cat,
    'selection_rule': 'select O1 only when both mean CV MAE and validation MAE improve by at least 1%',
    'selected_run': selected_run,
    'validation_consults': {'E02-full-reproduction': 1, 'O1-new': 1},
    'test_policy': 'test rows excluded before analysis; no test summaries, predictions, or scores',
    'library_versions': {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__, 'scikit-learn': sklearn.__version__},
    'output_sha256': {name: sha256(BUNDLE / name) for name in evidence_names},
}
(BUNDLE / 'run_receipt.json').write_text(json.dumps(receipt, indent=2) + '\n', encoding='utf-8')
print(json.dumps(receipt, indent=2))

{
  "workflow_run": "OCT-RN-FEATURE-REVIEW-001",
  "modeling_data_version": "OCT-MD-001",
  "split_version": "OCT-SPLIT-001",
  "seed": 42,
  "source_sha256": "74037cb248a1064e4d578692a4f4e5d8492ed1b2033daf643496e1b68b14ae03",
  "manifest_sha256": "de10b0e17ad56174a7b126352a0a41b3b3d477bbddd65f37b87a6235f582a64b",
  "notebook_code_sha256": "345e1feec736b4bb048c2c0bbc78262c5c185d5de355c31d950026eedad00aec",
  "code_version_commit": "0693f5d5a75569778590fdf081e0b1695e2b38b2",
  "git_dirty_at_start": false,
  "partition_counts_used": {
    "train": 131822,
    "validation": 28248,
    "test": "excluded"
  },
  "cv_folds": 5,
  "model_settings": {
    "loss": "absolute_error",
    "learning_rate": 0.1,
    "max_leaf_nodes": 63,
    "min_samples_leaf": 50,
    "max_iter": 200,
    "early_stopping": true,
    "random_state": 42,
    "rare_min_frequency": 132
  },
  "feature_counts": {
    "E02-full": 130,
    "O1": 99
  },
  "excluded_features": [
    "cat15",
    "cat17",
    "cat18",
    "